# ZGLMM159 — C1 — Análise Exploratória

**Tabela:** `prd_procurement.corp_curated.vw_zglmm159_c1`
**Ambiente:** 🔴 **PRODUÇÃO (PRD)** · **Objeto:** VIEW
**Domínio:** Avaliacao do material por centro (classe de avaliacao, controle de preco, preco medio)
**Colunas:** 9 · **Parte C1** de 3 (C1 · M1 · M2)

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** e **somente leitura** — nenhum `CREATE`,
`INSERT` ou gravação. Sem widget, sem view temporária, sem ordem obrigatória.

## Alertas de partida _(dos comentários do `DESCRIBE` e da validação anterior)_

| Severidade | Alerta |
|---|---|
| 🟡 MÉDIA | **Sem campo de tipo de avaliação.** Na validação anterior esse campo gerava linhas extras (split valuation). Confirmar se `material + centro` ficou único (seção 18.1). |
| ⚪ BAIXA | Nenhuma coluna declarada sem fonte. |
| 🟡 MÉDIA | **`ind_marcado_exclusao` preservado sem filtro** — a view traz também itens marcados para eliminação. Se a ZGLMM159 no SAP filtra eliminados, a comparação acusará sobra (seção 16). |
| 🟡 MÉDIA | **Nenhuma coluna de auditoria de carga** (sem `dateingest`/`dh_carga`) — não é possível saber a data do snapshot. Anotar data/hora da consulta. |
| ⚪ BAIXA | **Objeto é VIEW** — `DESCRIBE DETAIL`/`HISTORY` comentados para não quebrar o Run All. |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Distribuição por centro — escolha do cenário |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação** |
| 12 | Datas |
| 13 | Códigos e zeros à esquerda |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| **18** | **Análises específicas + cruzamento C1 × M1 × M2** |
| 19 | Amostra ampliada |
| 20 | Resumo |


## 1. Metadados

In [0]:
-- 1. ESTRUTURA
DESCRIBE EXTENDED prd_procurement.corp_curated.vw_zglmm159_c1;

In [0]:
-- 1.1 Detalhe e historico nao se aplicam a view.
-- DESCRIBE DETAIL prd_procurement.corp_curated.vw_zglmm159_c1;
-- DESCRIBE HISTORY prd_procurement.corp_curated.vw_zglmm159_c1 LIMIT 20;
SELECT 'Objeto e uma VIEW — os dois comandos acima nao se aplicam' AS observacao;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT cod_material) AS materiais,
       COUNT(DISTINCT cod_centro) AS centros,
       COUNT(DISTINCT cod_empresa) AS empresas,
       COUNT(DISTINCT CONCAT_WS('|', cod_material, cod_centro)) AS chaves_material_centro
FROM prd_procurement.corp_curated.vw_zglmm159_c1;

## 3. Distribuição por centro

Base para o cenário de teste (10 mil a 300 mil linhas). A 3.1 mostra os centros já usados
nos demais testes (4128, 4014) e o 4058 da validação anterior.

In [0]:
-- 3. DISTRIBUICAO POR CENTRO
SELECT COALESCE(NULLIF(trim(cod_centro),''),'(vazio)') AS cod_centro, COUNT(*) AS linhas,
       COUNT(DISTINCT cod_material) AS materiais,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_c1), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000 THEN 'CANDIDATO A CENARIO' ELSE '' END AS sugestao
FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY 1 ORDER BY linhas DESC LIMIT 60;

In [0]:
-- 3.1 CENTROS DOS DEMAIS TESTES
SELECT cod_centro, COUNT(*) AS linhas, COUNT(DISTINCT cod_material) AS materiais,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_c1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro IN ('4128', '4014', '4058')
GROUP BY cod_centro ORDER BY cod_centro;

## 4. Granularidade e chave real

In [0]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_c1),
g AS (
  SELECT 'cod_material' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material` FROM prd_procurement.corp_curated.vw_zglmm159_c1)
UNION ALL
  SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM prd_procurement.corp_curated.vw_zglmm159_c1)
UNION ALL
  SELECT 'cod_material + cod_centro + cod_classe_avaliacao' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_classe_avaliacao` FROM prd_procurement.corp_curated.vw_zglmm159_c1)
UNION ALL
  SELECT 'cod_material + cod_centro + cod_empresa' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_empresa` FROM prd_procurement.corp_curated.vw_zglmm159_c1)
)
SELECT g.chave, t.total AS linhas, g.distintos, ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA' ELSE 'NAO UNICA' END AS veredito
FROM g CROSS JOIN t ORDER BY linhas_por_chave;

## 5. Duplicidade

Pela chave esperada `cod_material + cod_centro`.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `cod_material`, `cod_centro`, COUNT(*) AS qtd FROM prd_procurement.corp_curated.vw_zglmm159_c1
GROUP BY `cod_material`, `cod_centro` HAVING COUNT(*) > 1 ORDER BY qtd DESC LIMIT 30;

In [0]:
-- 5.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH dup AS (SELECT `cod_material`, `cod_centro` FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY `cod_material`, `cod_centro` HAVING COUNT(*) > 1),
d AS (SELECT c.* FROM prd_procurement.corp_curated.vw_zglmm159_c1 c JOIN dup ON c.`cod_material` <=> dup.`cod_material` AND c.`cod_centro` <=> dup.`cod_centro`),
agg AS (SELECT `cod_material`, `cod_centro`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `cod_classe_avaliacao`) AS `cod_classe_avaliacao`,
         COUNT(DISTINCT `tp_determinacao_preco`) AS `tp_determinacao_preco`,
         COUNT(DISTINCT `ind_controle_preco`) AS `ind_controle_preco`,
         COUNT(DISTINCT `qt_unidade_preco`) AS `qt_unidade_preco`,
         COUNT(DISTINCT `vl_preco_medio_movel`) AS `vl_preco_medio_movel`,
         COUNT(DISTINCT `ind_marcado_exclusao`) AS `ind_marcado_exclusao`
  FROM d GROUP BY `cod_material`, `cod_centro`)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real' ELSE 'constante' END AS veredito
FROM (SELECT stack(7,
    'cod_empresa', MAX(`cod_empresa`),
    'cod_classe_avaliacao', MAX(`cod_classe_avaliacao`),
    'tp_determinacao_preco', MAX(`tp_determinacao_preco`),
    'ind_controle_preco', MAX(`ind_controle_preco`),
    'qt_unidade_preco', MAX(`qt_unidade_preco`),
    'vl_preco_medio_movel', MAX(`vl_preco_medio_movel`),
    'ind_marcado_exclusao', MAX(`ind_marcado_exclusao`)
  ) AS (coluna, max_valores_distintos) FROM agg)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Coluna 100% nula que **não** está na lista de "sem fonte" é erro de carga.

In [0]:
-- 6. PREENCHIMENTO
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_c1),
perf AS (SELECT stack(9,
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_classe_avaliacao', 'string', COUNT_IF(`cod_classe_avaliacao` IS NULL), COUNT_IF(`cod_classe_avaliacao` IS NOT NULL AND lower(trim(`cod_classe_avaliacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_classe_avaliacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_determinacao_preco', 'string', COUNT_IF(`tp_determinacao_preco` IS NULL), COUNT_IF(`tp_determinacao_preco` IS NOT NULL AND lower(trim(`tp_determinacao_preco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_determinacao_preco`) RLIKE '^0+([.,]0+)?$'),
    'ind_controle_preco', 'string', COUNT_IF(`ind_controle_preco` IS NULL), COUNT_IF(`ind_controle_preco` IS NOT NULL AND lower(trim(`ind_controle_preco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_controle_preco`) RLIKE '^0+([.,]0+)?$'),
    'qt_unidade_preco', 'decimal(5,0)', COUNT_IF(`qt_unidade_preco` IS NULL), 0L, COUNT_IF(`qt_unidade_preco` = 0),
    'vl_preco_medio_movel', 'decimal(18,2)', COUNT_IF(`vl_preco_medio_movel` IS NULL), 0L, COUNT_IF(`vl_preco_medio_movel` = 0),
    'ind_marcado_exclusao', 'string', COUNT_IF(`ind_marcado_exclusao` IS NULL), COUNT_IF(`ind_marcado_exclusao` IS NOT NULL AND lower(trim(`ind_marcado_exclusao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_marcado_exclusao`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros) FROM prd_procurement.corp_curated.vw_zglmm159_c1)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0 THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END AS veredito,
       CASE WHEN p.coluna IN ('') THEN 'declarada sem fonte' ELSE '' END AS observacao
FROM perf p CROSS JOIN t ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

In [0]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_c1),
card AS (SELECT stack(9,
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'cod_classe_avaliacao', 'string', approx_count_distinct(`cod_classe_avaliacao`),
    'tp_determinacao_preco', 'string', approx_count_distinct(`tp_determinacao_preco`),
    'ind_controle_preco', 'string', approx_count_distinct(`ind_controle_preco`),
    'qt_unidade_preco', 'decimal(5,0)', approx_count_distinct(`qt_unidade_preco`),
    'vl_preco_medio_movel', 'decimal(18,2)', approx_count_distinct(`vl_preco_medio_movel`),
    'ind_marcado_exclusao', 'string', approx_count_distinct(`ind_marcado_exclusao`)
  ) AS (coluna, tipo, distintos) FROM prd_procurement.corp_curated.vw_zglmm159_c1)
SELECT c.coluna, c.tipo, c.distintos, ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1 THEN '1. CONSTANTE' WHEN c.distintos <= 3 THEN '2. muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador' ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t ORDER BY c.distintos;

## 8. Domínio das categóricas

Top 8 valores de cada uma.

In [0]:
-- 8. DOMINIO
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_classe_avaliacao' AS coluna, CAST(`cod_classe_avaliacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY `cod_classe_avaliacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_determinacao_preco' AS coluna, CAST(`tp_determinacao_preco` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY `tp_determinacao_preco` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_controle_preco' AS coluna, CAST(`ind_controle_preco` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY `ind_controle_preco` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_marcado_exclusao' AS coluna, CAST(`ind_marcado_exclusao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY `ind_marcado_exclusao` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil numérico

Todos `decimal` — conciliação exata, sem tolerância.

In [0]:
-- 10. PERFIL NUMERICO
SELECT * FROM (SELECT stack(2,
    'qt_unidade_preco', 'decimal(5,0)', COUNT(`qt_unidade_preco`), CAST(MIN(`qt_unidade_preco`) AS DOUBLE), CAST(MAX(`qt_unidade_preco`) AS DOUBLE), CAST(AVG(`qt_unidade_preco`) AS DOUBLE), CAST(percentile_approx(`qt_unidade_preco`, 0.5) AS DOUBLE), COUNT_IF(`qt_unidade_preco` < 0), COUNT_IF(`qt_unidade_preco` = 0),
    'vl_preco_medio_movel', 'decimal(18,2)', COUNT(`vl_preco_medio_movel`), CAST(MIN(`vl_preco_medio_movel`) AS DOUBLE), CAST(MAX(`vl_preco_medio_movel`) AS DOUBLE), CAST(AVG(`vl_preco_medio_movel`) AS DOUBLE), CAST(percentile_approx(`vl_preco_medio_movel`, 0.5) AS DOUBLE), COUNT_IF(`vl_preco_medio_movel` < 0), COUNT_IF(`vl_preco_medio_movel` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros) FROM prd_procurement.corp_curated.vw_zglmm159_c1)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

In [0]:
-- 11. TOTAIS
SELECT * FROM (SELECT stack(2,
    'qt_unidade_preco', CAST(ROUND(SUM(`qt_unidade_preco`), 3) AS STRING), COUNT(`qt_unidade_preco`),
    'vl_preco_medio_movel', CAST(ROUND(SUM(`vl_preco_medio_movel`), 3) AS STRING), COUNT(`vl_preco_medio_movel`)
  ) AS (coluna, total, linhas_preenchidas) FROM prd_procurement.corp_curated.vw_zglmm159_c1)
ORDER BY coluna;

## 12. Datas

In [0]:
-- 12. DATAS
SELECT 'Esta view nao possui colunas de data' AS observacao;

## 13. Códigos e zeros à esquerda

Na validação anterior, `cod_material` vinha com zeros à esquerda e o extrato SAP sem — normalizar os dois lados.

In [0]:
-- 13. CODIGOS
SELECT coluna, vazios, len_min, len_max, com_zeros_esq, distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes
FROM (SELECT stack(4,
    'cod_material', COUNT_IF(`cod_material` IS NULL OR trim(`cod_material`) = ''), MIN(length(trim(`cod_material`))), MAX(length(trim(`cod_material`))), COUNT_IF(trim(`cod_material`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_material`)), COUNT(DISTINCT regexp_replace(trim(`cod_material`), '^0+', '')),
    'cod_centro', COUNT_IF(`cod_centro` IS NULL OR trim(`cod_centro`) = ''), MIN(length(trim(`cod_centro`))), MAX(length(trim(`cod_centro`))), COUNT_IF(trim(`cod_centro`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_centro`)), COUNT(DISTINCT regexp_replace(trim(`cod_centro`), '^0+', '')),
    'cod_empresa', COUNT_IF(`cod_empresa` IS NULL OR trim(`cod_empresa`) = ''), MIN(length(trim(`cod_empresa`))), MAX(length(trim(`cod_empresa`))), COUNT_IF(trim(`cod_empresa`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_empresa`)), COUNT(DISTINCT regexp_replace(trim(`cod_empresa`), '^0+', '')),
    'cod_classe_avaliacao', COUNT_IF(`cod_classe_avaliacao` IS NULL OR trim(`cod_classe_avaliacao`) = ''), MIN(length(trim(`cod_classe_avaliacao`))), MAX(length(trim(`cod_classe_avaliacao`))), COUNT_IF(trim(`cod_classe_avaliacao`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_classe_avaliacao`)), COUNT(DISTINCT regexp_replace(trim(`cod_classe_avaliacao`), '^0+', ''))
  ) AS (coluna, vazios, len_min, len_max, com_zeros_esq, distintos_bruto, distintos_sem_zeros) FROM prd_procurement.corp_curated.vw_zglmm159_c1)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, há registros 100% idênticos.

In [0]:
-- 14. CHECKSUM
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_classe_avaliacao` AS STRING), ''), COALESCE(CAST(`tp_determinacao_preco` AS STRING), ''), COALESCE(CAST(`ind_controle_preco` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`vl_preco_medio_movel` AS STRING), ''), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '')))) AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_classe_avaliacao` AS STRING), ''), COALESCE(CAST(`tp_determinacao_preco` AS STRING), ''), COALESCE(CAST(`ind_controle_preco` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`vl_preco_medio_movel` AS STRING), ''), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '')))) AS linhas_100pct_identicas
FROM prd_procurement.corp_curated.vw_zglmm159_c1;

## 15. Amostra

In [0]:
-- 15. AMOSTRA
SELECT * FROM prd_procurement.corp_curated.vw_zglmm159_c1
ORDER BY `cod_material`, `cod_centro`
LIMIT 20;

## 16. Distribuição interna

Inclui `ind_marcado_exclusao` — mostra quanto da view são itens marcados para eliminação.

In [0]:
-- 16. DISTRIBUICAO POR cod_empresa
SELECT COALESCE(NULLIF(trim(`cod_empresa`), ''), '(vazio)') AS `cod_empresa`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_c1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR ind_controle_preco
SELECT COALESCE(NULLIF(trim(`ind_controle_preco`), ''), '(vazio)') AS `ind_controle_preco`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_c1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_classe_avaliacao
SELECT COALESCE(NULLIF(trim(`cod_classe_avaliacao`), ''), '(vazio)') AS `cod_classe_avaliacao`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_c1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR ind_marcado_exclusao
SELECT COALESCE(NULLIF(trim(`ind_marcado_exclusao`), ''), '(vazio)') AS `ind_marcado_exclusao`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_c1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

## 18. Análises específicas — C1 (avaliação)

### 18.1 Split valuation

Na validação anterior (SAP → DEV, centro 4058), o campo de tipo de avaliação gerava **linhas extras**
para materiais com avaliação dividida (ex.: material 260762). A C1 não tem mais esse campo —
esta célula confirma se `material + centro` ficou único.

In [0]:
-- 18.1 LINHAS POR MATERIAL + CENTRO
SELECT linhas_por_chave, COUNT(*) AS chaves
FROM (SELECT cod_material, cod_centro, COUNT(*) AS linhas_por_chave FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY cod_material, cod_centro)
GROUP BY linhas_por_chave ORDER BY linhas_por_chave;

In [0]:
-- 18.1b EXEMPLOS (se houver repeticao)
SELECT cod_material, cod_centro, COUNT(*) AS linhas,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(cod_classe_avaliacao))) AS classes,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(ind_controle_preco))) AS controles
FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY cod_material, cod_centro HAVING COUNT(*) > 1
ORDER BY linhas DESC LIMIT 20;

### 18.2 Controle de preço × preço médio

`ind_controle_preco = V` (preço médio móvel) deveria ter `vl_preco_medio_movel` preenchido e positivo.
`S` (preço padrão) pode ter o médio zerado.

In [0]:
-- 18.2 CONTROLE DE PRECO x PRECO MEDIO
SELECT COALESCE(NULLIF(trim(ind_controle_preco),''),'(vazio)') AS ind_controle_preco,
       COUNT(*) AS linhas,
       COUNT_IF(vl_preco_medio_movel IS NULL) AS preco_nulo,
       COUNT_IF(vl_preco_medio_movel = 0) AS preco_zero,
       COUNT_IF(vl_preco_medio_movel < 0) AS preco_negativo,
       COUNT_IF(qt_unidade_preco IS NULL OR qt_unidade_preco <= 0) AS unidade_preco_invalida,
       ROUND(AVG(vl_preco_medio_movel), 2) AS preco_medio
FROM prd_procurement.corp_curated.vw_zglmm159_c1 GROUP BY 1 ORDER BY linhas DESC;

### 18.9 🔗 Cruzamento entre as 3 views

A ZGLMM159 foi dividida em C1, M1 e M2. As três deveriam cobrir o **mesmo universo de
`material + centro`** e concordar na empresa. Esta célula mede a cobertura cruzada — divergência
aqui indica material que vai "sumir" quando as três forem reunidas no relatório final.

> Na validação anterior, **434 materiais existiam no SAP e não no Datalake**. Esta análise
> ajuda a localizar em qual das três views eles se perdem.

In [0]:
-- 18.9 COBERTURA DE MATERIAL + CENTRO ENTRE C1, M1, M2
WITH c1 AS (SELECT DISTINCT cod_material, cod_centro FROM prd_procurement.corp_curated.vw_zglmm159_c1),
     m1 AS (SELECT DISTINCT cod_material, cod_centro FROM prd_procurement.corp_curated.vw_zglmm159_m1),
     m2 AS (SELECT DISTINCT cod_material, cod_centro FROM prd_procurement.corp_curated.vw_zglmm159_m2),
     u  AS (SELECT * FROM c1 UNION SELECT * FROM m1 UNION SELECT * FROM m2),
     x  AS (SELECT u.cod_material, u.cod_centro,
                   c1.cod_material IS NOT NULL AS em_c1,
                   m1.cod_material IS NOT NULL AS em_m1,
                   m2.cod_material IS NOT NULL AS em_m2
            FROM u
            LEFT JOIN c1 ON u.cod_material = c1.cod_material AND u.cod_centro = c1.cod_centro
            LEFT JOIN m1 ON u.cod_material = m1.cod_material AND u.cod_centro = m1.cod_centro
            LEFT JOIN m2 ON u.cod_material = m2.cod_material AND u.cod_centro = m2.cod_centro)
SELECT CONCAT(CASE WHEN em_c1 THEN 'C1 ' ELSE '-- ' END,
              CASE WHEN em_m1 THEN 'M1 ' ELSE '-- ' END,
              CASE WHEN em_m2 THEN 'M2'  ELSE '--' END) AS presenca,
       COUNT(*) AS material_centro,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM x GROUP BY 1 ORDER BY material_centro DESC;

In [0]:
-- 18.9b EMPRESA E INDICADOR DE EXCLUSAO CONSISTENTES ENTRE AS VIEWS
WITH a AS (SELECT DISTINCT cod_material, cod_centro, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_c1),
     b AS (SELECT DISTINCT cod_material, cod_centro, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_m1),
     c AS (SELECT DISTINCT cod_material, cod_centro, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_m2)
SELECT 'C1 x M1' AS par,
       COUNT(*) AS material_centro_em_comum,
       COUNT_IF(COALESCE(a.cod_empresa,'') <> COALESCE(b.cod_empresa,'')) AS empresa_diverge,
       COUNT_IF(COALESCE(a.ind_marcado_exclusao,'') <> COALESCE(b.ind_marcado_exclusao,'')) AS exclusao_diverge
  FROM a JOIN b ON a.cod_material = b.cod_material AND a.cod_centro = b.cod_centro
UNION ALL
SELECT 'C1 x M2',
       COUNT(*),
       COUNT_IF(COALESCE(a.cod_empresa,'') <> COALESCE(c.cod_empresa,'')),
       COUNT_IF(COALESCE(a.ind_marcado_exclusao,'') <> COALESCE(c.ind_marcado_exclusao,''))
  FROM a JOIN c ON a.cod_material = c.cod_material AND a.cod_centro = c.cod_centro;

## 19. Amostra ampliada

A extração completa fica nos notebooks de cenário.

In [0]:
-- 19. AMOSTRA AMPLIADA
SELECT * FROM prd_procurement.corp_curated.vw_zglmm159_c1
ORDER BY `cod_material`, `cod_centro`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_c1),
g AS (
  SELECT 'cod_material' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material` FROM prd_procurement.corp_curated.vw_zglmm159_c1)
UNION ALL
  SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM prd_procurement.corp_curated.vw_zglmm159_c1)
UNION ALL
  SELECT 'cod_material + cod_centro + cod_classe_avaliacao' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_classe_avaliacao` FROM prd_procurement.corp_curated.vw_zglmm159_c1)
UNION ALL
  SELECT 'cod_material + cod_centro + cod_empresa' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_empresa` FROM prd_procurement.corp_curated.vw_zglmm159_c1)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'ZGLMM159 - C1' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'ambiente', 'PRD'
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'vw_zglmm159_c1'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '9 (0 declaradas sem fonte)'
UNION ALL SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ', CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total), 'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Rodar os **3 notebooks** (C1, M1, M2) e comparar a seção 18.9 entre eles.
2. Escolher o recorte de teste pela **seção 3/3.1**.
3. Gerar os notebooks de cenário e extrair a ZGLMM159 no **SAP PRD** com o mesmo recorte e na mesma data.

### Checklist antes de comparar

- [ ] Chave real confirmada (seção 4)
- [ ] Colunas sem fonte excluídas da comparação (seção 6.1)
- [ ] Zeros à esquerda normalizados (seção 13)
- [ ] Itens marcados para exclusão tratados conforme o filtro do SAP (seção 16)
- [ ] Cobertura C1 × M1 × M2 conferida (seção 18.9)
